# 🎨 Módulo 05 · Seaborn

Visualización estadística orientada al análisis: pregunta → tipo de dato → visualización adecuada → interpretación.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m05)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **Seaborn: gráficos estadísticos desde tablas**: Hacer gráficos **estadísticos** (distribuciones, comparaciones entre grupos, relaciones) directamente desde un DataFrame, con poco código y buen diseño por defecto.
- **Distribuciones**: Ver cómo se distribuye una variable numérica: valores típicos, dispersión, asimetría, outliers; y comparar distribuciones entre grupos.
- **Relaciones**: Estudiar cómo se relacionan dos variables numéricas (o una numérica con el tiempo).
- **Comparar grupos (categorías)**: Comparar una variable numérica entre categorías, o contar cuántas observaciones hay de cada categoría.
- **Tercera variable con hue**: Agregar una tercera variable (categórica) a un gráfico mediante colores, para comparar subgrupos.
- **Correlación**: Resumir de un vistazo qué variables numéricas se mueven juntas (y en qué dirección).
- **EDA visual: de la pregunta al hallazgo**: Explorar un dataset con gráficos de forma ordenada: cada gráfico responde una pregunta y genera un hallazgo (o una nueva pregunta).

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n_rng = np.random.default_rng(7)\n_n = 120\npropinas = pd.DataFrame({\n    "cuenta": np.round(_rng.gamma(4, 5, _n) + 5, 2),\n    "dia": _rng.choice(["Jue", "Vie", "Sáb", "Dom"], _n, p=[0.2, 0.2, 0.35, 0.25]),\n    "momento": _rng.choice(["Almuerzo", "Cena"], _n, p=[0.35, 0.65]),\n    "fumador": _rng.choice(["Sí", "No"], _n, p=[0.4, 0.6]),\n    "personas": _rng.integers(1, 7, _n),\n})\npropinas["propina"] = np.round(propinas["cuenta"] * _rng.uniform(0.08, 0.22, _n) + 0.5, 2)\ndel _rng, _n\n'
_SOL_B64 = 'eyJzbnMwMSI6IHsic29sdXRpb24iOiAic25zLmhpc3RwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInaGlzdHBsb3QnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkucGF0Y2hlcykgPiAwIiwgImN1c3RvbV9tc2ciOiAiVXNhIHNucy5oaXN0cGxvdCBjb24gbGEgY29sdW1uYSAnY3VlbnRhJy4ifSwgImhpbnRzIjogWyJVbmEgdmFyaWFibGUgbnVtw6lyaWNhOiDCv3F1w6kgZ3LDoWZpY28/IiwgIkVuIHNlYWJvcm4gc2UgcGFzYSBsYSB0YWJsYSBjb24gZGF0YT0geSBsYSBjb2x1bW5hIGNvbiB4PS4iLCAic25zLmhpc3RwbG90KGRhdGE9Li4uLCB4PS4uLikuIl0sICJleHBsYWluIjogImBoaXN0cGxvdGAgY3VlbnRhIGN1w6FudGFzIGN1ZW50YXMgY2FlbiBlbiBjYWRhIGludGVydmFsbyBkZSBtb250bywgdG9tYW5kbyBsYSBjb2x1bW5hIGRpcmVjdGFtZW50ZSBkZWwgRGF0YUZyYW1lLiIsICJwc2V1ZG8iOiAiaGlzdHBsb3QoZGF0YT1wcm9waW5hcywgeD0nY3VlbnRhJykifSwgInNuczAyIjogeyJzb2x1dGlvbiI6ICJzbnMuaGlzdHBsb3QoZGF0YT1wcm9waW5hcywgeD0ncHJvcGluYScsIGJpbnM9MTUsIGtkZT1UcnVlKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLnBhdGNoZXMpID09IDE1IGFuZCBsZW4oX2F4KCkubGluZXMpID49IDEiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSAxNSBiYXJyYXMgeSBsYSBjdXJ2YSBLREUuIn0sICJoaW50cyI6IFsiTWlzbW8gdGlwbyBkZSBncsOhZmljbywgY29uIGRvcyBhZ3JlZ2Fkb3MuIiwgIlNlIHB1ZWRlIGZpamFyIGxhIGNhbnRpZGFkIGRlIGludGVydmFsb3MgeSBzdXBlcnBvbmVyIHVuYSBjdXJ2YSBzdWF2aXphZGEuIiwgImJpbnM9MTUgeSBrZGU9VHJ1ZS4iXSwgImV4cGxhaW4iOiAiTGEgY3VydmEgS0RFIGVzIHVuYSB2ZXJzacOzbiBzdWF2aXphZGEgZGVsIGhpc3RvZ3JhbWEgcXVlIGF5dWRhIGEgdmVyIGxhIGZvcm1hIGdlbmVyYWwgZGUgbGEgZGlzdHJpYnVjacOzbi4iLCAicHNldWRvIjogImhpc3RwbG90KGRhdGE9cHJvcGluYXMsIHg9J3Byb3BpbmEnLCAxNSBpbnRlcnZhbG9zLCBjb24gY3VydmEgS0RFKSJ9LCAic25zMDMiOiB7InNvbHV0aW9uIjogInNucy5rZGVwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScsIGh1ZT0nbW9tZW50bycsIGZpbGw9VHJ1ZSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidrZGVwbG90JyBpbiBfcHFfY29kZSBhbmQgc29ydGVkKF9sZWdlbmQoKSkgPT0gWydBbG11ZXJ6bycsICdDZW5hJ10iLCAiY3VzdG9tX21zZyI6ICJVc2Egc25zLmtkZXBsb3QgY29uIGh1ZT0nbW9tZW50bycgKGxhIGxleWVuZGEgZGViZSBtb3N0cmFyIEFsbXVlcnpvIHkgQ2VuYSkuIn0sICJoaW50cyI6IFsiQ29tcGFyw6FzIGxhIG1pc21hIHZhcmlhYmxlIGVudHJlIGRvcyBncnVwb3MuIiwgIlVuYSBjdXJ2YSBwb3IgZ3J1cG8sIHN1cGVycHVlc3RhcywgY29uIGNvbG9yZXMuIiwgImtkZXBsb3QgY29uIGh1ZT0nbW9tZW50bycgeSBmaWxsPVRydWUuIl0sICJleHBsYWluIjogImBrZGVwbG90YCBjb24gYGh1ZWAgZGlidWphIHVuYSBjdXJ2YSBwb3IgZ3J1cG86IHNlIHB1ZWRlIGNvbXBhcmFyIGTDs25kZSBzZSBjb25jZW50cmFuIGxhcyBjdWVudGFzIGRlIGFsbXVlcnpvIHZzIGNlbmEuIiwgInBzZXVkbyI6ICJrZGVwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScsIHNlcGFyYWRvIHBvciBtb21lbnRvLCByZWxsZW5vKSJ9LCAic25zLWUwMSI6IHsic29sdXRpb24iOiAic25zLmhpc3RwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScsIGh1ZT0nbW9tZW50bycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInaGlzdHBsb3QnIGluIF9wcV9jb2RlIGFuZCBzb3J0ZWQoX2xlZ2VuZCgpKSA9PSBbJ0FsbXVlcnpvJywgJ0NlbmEnXSIsICJjdXN0b21fbXNnIjogIlVzw6Egc25zLmhpc3RwbG90IGNvbiBkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnIHkgaHVlPSdtb21lbnRvJyAobGEgbGV5ZW5kYSBkZWJlIG1vc3RyYXIgQWxtdWVyem8geSBDZW5hKS4ifSwgImhpbnRzIjogWyJMZcOpIGVsIGVycm9yOiDCv3F1w6kgdmFsb3Igbm8gcHVlZGUgaW50ZXJwcmV0YXI/IiwgIsK/RGUgZMOzbmRlIHNhY2Fyw61hIHNlYWJvcm4gbGEgY29sdW1uYSAnbW9tZW50byc/IiwgIlBhc8OhIGxhIHRhYmxhIGNvbXBsZXRhIGNvbiBkYXRhPSB5IGxhcyBjb2x1bW5hcyBwb3Igbm9tYnJlLiJdLCAiZXhwbGFpbiI6ICJgaHVlPSdtb21lbnRvJ2AgZXMgZWwgKipub21icmUqKiBkZSB1bmEgY29sdW1uYTsgc2VhYm9ybiBsYSBidXNjYSBlbiBlbCBEYXRhRnJhbWUgcGFzYWRvIGVuIGBkYXRhPWAuIENvbW8gc29sbyByZWNpYmnDsyBsYSBTZXJpZXMgYGN1ZW50YWAsIG5vIHB1ZWRlIGludGVycHJldGFyICdtb21lbnRvJyB5IGxhbnphIHVuIGVycm9yLiBQYXNhbmRvIGBkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnYCB0b2RvIHNlIHJlc3VlbHZlIHBvciBub21icmUuIiwgInBzZXVkbyI6ICJoaXN0cGxvdChkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnLCBodWU9J21vbWVudG8nKSJ9LCAic25zMDQiOiB7InNvbHV0aW9uIjogInNucy5zY2F0dGVycGxvdChkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnLCB5PSdwcm9waW5hJywgaHVlPSdtb21lbnRvJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidzY2F0dGVycGxvdCcgaW4gX3BxX2NvZGUgYW5kIHN1bShsZW4oYy5nZXRfb2Zmc2V0cygpKSBmb3IgYyBpbiBfYXgoKS5jb2xsZWN0aW9ucykgPT0gMTIwIGFuZCB7J0FsbXVlcnpvJywgJ0NlbmEnfSA8PSBzZXQoX2xlZ2VuZCgpKSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHZlcnNlIGxvcyAxMjAgcHVudG9zLCBjb2xvcmVhZG9zIHBvciAnbW9tZW50bycuIn0sICJoaW50cyI6IFsiRG9zIG51bcOpcmljYXMgKGN1ZW50YSwgcHJvcGluYSkgKyB1bmEgY2F0ZWfDs3JpY2EgcGFyYSBjb2xvcmVhci4iLCAic2NhdHRlcnBsb3QgY29uIHgsIHkgeSBodWUuIiwgInNucy5zY2F0dGVycGxvdChkYXRhPS4uLiwgeD0uLi4sIHk9Li4uLCBodWU9Li4uKS4iXSwgImV4cGxhaW4iOiAiQ2FkYSBwdW50byBlcyB1bmEgY3VlbnRhOyBlbCBjb2xvciBpbmRpY2EgZWwgbW9tZW50by4gUGVybWl0ZSB2ZXIgbGEgcmVsYWNpw7NuIHkgc2kgZGlmaWVyZSBlbnRyZSBncnVwb3MuIiwgInBzZXVkbyI6ICJzY2F0dGVycGxvdChkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnLCB5PSdwcm9waW5hJywgY29sb3IgcG9yIG1vbWVudG8pIn0sICJzbnMwNSI6IHsic29sdXRpb24iOiAic25zLnJlZ3Bsb3QoZGF0YT1wcm9waW5hcywgeD0nY3VlbnRhJywgeT0ncHJvcGluYScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICIncmVncGxvdCcgaW4gX3BxX2NvZGUgYW5kIGxlbihfYXgoKS5saW5lcykgPj0gMSBhbmQgbGVuKF9heCgpLmNvbGxlY3Rpb25zKSA+PSAxIiwgImN1c3RvbV9tc2ciOiAiVXNhIHNucy5yZWdwbG90IGNvbiBjdWVudGEgZW4gWCB5IHByb3BpbmEgZW4gWS4ifSwgImhpbnRzIjogWyJRdWVyw6lzIGxhIHJlbGFjacOzbiB5IGFkZW3DoXMgbGEgdGVuZGVuY2lhLiIsICJIYXkgdW4gZ3LDoWZpY28gcXVlIGFncmVnYSB1bmEgcmVjdGEgZGUgcmVncmVzacOzbiBhIGxvcyBwdW50b3MuIiwgInNucy5yZWdwbG90LiJdLCAiZXhwbGFpbiI6ICJgcmVncGxvdGAgZGlidWphIGxvcyBwdW50b3MgeSB1bmEgcmVjdGEgZGUgdGVuZGVuY2lhIGNvbiBzdSBpbnRlcnZhbG8gZGUgY29uZmlhbnphLiIsICJwc2V1ZG8iOiAicmVncGxvdChkYXRhPXByb3BpbmFzLCB4PSdjdWVudGEnLCB5PSdwcm9waW5hJykifSwgInNucy1nMDEiOiB7InNvbHV0aW9uIjogInNucy5zY2F0dGVycGxvdChkYXRhPXByb3BpbmFzLCB4PSdwZXJzb25hcycsIHk9J2N1ZW50YScsIGh1ZT0nZnVtYWRvcicpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInc2NhdHRlcnBsb3QnIGluIF9wcV9jb2RlIGFuZCBzdW0obGVuKGMuZ2V0X29mZnNldHMoKSkgZm9yIGMgaW4gX2F4KCkuY29sbGVjdGlvbnMpID09IDEyMCBhbmQgeydObycsICdTw60nfSA8PSBzZXQoX2xlZ2VuZCgpKSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHZlcnNlIGxvcyAxMjAgcHVudG9zIChwZXJzb25hcyB2cyBjdWVudGEpLCBjb2xvcmVhZG9zIHBvciBmdW1hZG9yLiJ9LCAiaGludHMiOiBbIsK/UXXDqSBjYW1iaWE6IGVsIHRpcG8gZGUgZ3LDoWZpY28gbyBsYXMgdmFyaWFibGVzPyIsICJNaXNtYSBmdW5jacOzbiwgb3RyYXMgY29sdW1uYXMuIiwgIng9J3BlcnNvbmFzJywgeT0nY3VlbnRhJywgaHVlPSdmdW1hZG9yJy4iXSwgImV4cGxhaW4iOiAiTWlzbW8gZ3LDoWZpY28gKHJlbGFjacOzbiBlbnRyZSBkb3MgbnVtw6lyaWNhcyArIGNvbG9yIHBvciBncnVwbyksIGFwbGljYWRvIGEgb3RyYSBwcmVndW50YS4gTm90w6EgcXVlIHBlcnNvbmFzIHRvbWEgcG9jb3MgdmFsb3JlczogbG9zIHB1bnRvcyBxdWVkYW4gZW4gY29sdW1uYXMuIiwgInBzZXVkbyI6ICJzY2F0dGVycGxvdChkYXRhPXByb3BpbmFzLCB4PSdwZXJzb25hcycsIHk9J2N1ZW50YScsIGh1ZT0nZnVtYWRvcicpIn0sICJzbnMwNiI6IHsic29sdXRpb24iOiAic25zLnNldF90aGVtZShzdHlsZT0nd2hpdGVncmlkJylcbnNucy5saW5lcGxvdChkYXRhPXZlbnRhc19kaWFyaWFzLCB4PSdmZWNoYScsIHk9J3ZlbnRhcycpXG5wbHQudGl0bGUoJ1RlbmRlbmNpYSBkZSB2ZW50YXMnKVxucGx0Lnh0aWNrcyhyb3RhdGlvbj00NSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidsaW5lcGxvdCcgaW4gX3BxX2NvZGUgYW5kICd3aGl0ZWdyaWQnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkubGluZXMpID49IDEgYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdUZW5kZW5jaWEgZGUgdmVudGFzJyIsICJjdXN0b21fbXNnIjogIlVzYSBzZXRfdGhlbWUoc3R5bGU9J3doaXRlZ3JpZCcpLCBzbnMubGluZXBsb3QgeSBlbCB0w610dWxvICdUZW5kZW5jaWEgZGUgdmVudGFzJy4ifSwgImhpbnRzIjogWyJFcyB1bmEgZXZvbHVjacOzbiBlbiBlbCB0aWVtcG8uIiwgIlByaW1lcm8gZWwgZXN0aWxvLCBkZXNwdcOpcyBlbCBncsOhZmljbywgZGVzcHXDqXMgZWwgdMOtdHVsby4iLCAic25zLnNldF90aGVtZShzdHlsZT0nd2hpdGVncmlkJyksIHNucy5saW5lcGxvdCwgcGx0LnRpdGxlLiJdLCAiZXhwbGFpbiI6ICJgc2V0X3RoZW1lYCBkZWZpbmUgZWwgZXN0aWxvIGdlbmVyYWw7IGBsaW5lcGxvdGAgZGlidWphIGxhIGV2b2x1Y2nDs24geSBlbCB0w610dWxvIHNlIGFncmVnYSBjb24gbWF0cGxvdGxpYiAoc2VhYm9ybiBzZSBjb21iaW5hIGNvbiBwbHQpLiIsICJwc2V1ZG8iOiAiZXN0aWxvICd3aGl0ZWdyaWQnXG5saW5lcGxvdChkYXRhPXZlbnRhc19kaWFyaWFzLCB4PSdmZWNoYScsIHk9J3ZlbnRhcycpXG50w610dWxvICdUZW5kZW5jaWEgZGUgdmVudGFzJyJ9LCAic25zMDciOiB7InNvbHV0aW9uIjogInNucy5ib3hwbG90KGRhdGE9cHJvcGluYXMsIHg9J2RpYScsIHk9J3Byb3BpbmEnLCBvcmRlcj1bJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidib3hwbG90JyBpbiBfcHFfY29kZSBhbmQgX3h0aWNrcygpID09IFsnSnVlJywgJ1ZpZScsICdTw6FiJywgJ0RvbSddIiwgImN1c3RvbV9tc2ciOiAiVXNhIHNucy5ib3hwbG90IGNvbiB4PSdkaWEnLCB5PSdwcm9waW5hJyB5IGVsIG9yZGVuIGRlIGTDrWFzIGluZGljYWRvLiJ9LCAiaGludHMiOiBbIkNvbXBhcsOhcyBsYSBkaXN0cmlidWNpw7NuIGRlIHVuYSBudW3DqXJpY2EgZW50cmUgZMOtYXMuIiwgIkJveHBsb3Q6IGNhdGVnw7NyaWNhIGVuIFgsIG51bcOpcmljYSBlbiBZLiIsICJvcmRlcj0gZmlqYSBlbCBvcmRlbiBkZSBsb3MgZMOtYXMuIl0sICJleHBsYWluIjogIkVsIGJveHBsb3QgbXVlc3RyYSwgcGFyYSBjYWRhIGTDrWEsIGxhIG1lZGlhbmEsIGxvcyBjdWFydGlsZXMgeSBsb3Mgb3V0bGllcnMgZGUgbGFzIHByb3BpbmFzLCBlbiBlbCBvcmRlbiBpbmRpY2Fkby4iLCAicHNldWRvIjogImJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nZGlhJywgeT0ncHJvcGluYScsIG9yZGVuIEp1ZSDihpIgRG9tKSJ9LCAic25zLWcwMiI6IHsic29sdXRpb24iOiAic25zLmJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycsIHk9J2N1ZW50YScsIG9yZGVyPVsnQWxtdWVyem8nLCAnQ2VuYSddKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2JveHBsb3QnIGluIF9wcV9jb2RlIGFuZCBfeHRpY2tzKCkgPT0gWydBbG11ZXJ6bycsICdDZW5hJ10iLCAiY3VzdG9tX21zZyI6ICJCb3hwbG90IGRlIGN1ZW50YSBwb3IgbW9tZW50byBjb24gZWwgb3JkZW4gWydBbG11ZXJ6bycsICdDZW5hJ10uIn0sICJoaW50cyI6IFsiwr9RdcOpIGNhbWJpYSByZXNwZWN0byBkZWwgYm94cGxvdCBwb3IgZMOtYT8iLCAiQ2FtYmlhbiBsYSB2YXJpYWJsZSBkZSBYLCBsYSBkZSBZIHkgZWwgb3JkZW4uIiwgIng9J21vbWVudG8nLCB5PSdjdWVudGEnLCBvcmRlcj1bLi4uXS4iXSwgImV4cGxhaW4iOiAiTWlzbW8gcGF0csOzbiBjYXRlZ8OzcmljYSArIG51bcOpcmljYSDihpIgYm94cGxvdDsgY2FtYmlhbiBsYXMgY29sdW1uYXMgeSBlbCBvcmRlbi4iLCAicHNldWRvIjogImJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycsIHk9J2N1ZW50YScsIG9yZGVuIFtBbG11ZXJ6bywgQ2VuYV0pIn0sICJzbnMwOCI6IHsic29sdXRpb24iOiAic25zLmJhcnBsb3QoZGF0YT1wcm9waW5hcywgeD0nZGlhJywgeT0ncHJvcGluYScsIG9yZGVyPVsnSnVlJywgJ1ZpZScsICdTw6FiJywgJ0RvbSddLCBlcnJvcmJhcj1Ob25lKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2JhcnBsb3QnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkucGF0Y2hlcykgPT0gNCBhbmQgbnAuYWxsY2xvc2UoX2hlaWdodHMoKSwgcHJvcGluYXMuZ3JvdXBieSgnZGlhJylbJ3Byb3BpbmEnXS5tZWFuKCkucmVpbmRleChbJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSkudmFsdWVzKSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHZlcnNlIDQgYmFycmFzIGNvbiBsYSBwcm9waW5hIG1lZGlhIGRlIGNhZGEgZMOtYSwgZW4gZWwgb3JkZW4gcGVkaWRvLiJ9LCAiaGludHMiOiBbIkFob3JhIGltcG9ydGEgZWwgcHJvbWVkaW8gcG9yIGTDrWEsIG5vIGxhIGRpc3RyaWJ1Y2nDs24uIiwgImJhcnBsb3QgY2FsY3VsYSBlbCBwcm9tZWRpbyBwb3IgZ3J1cG8gYXV0b23DoXRpY2FtZW50ZS4iLCAib3JkZXI9IHkgZXJyb3JiYXI9Tm9uZSBwYXJhIHF1aXRhciBsYXMgYmFycmFzIGRlIGVycm9yLiJdLCAiZXhwbGFpbiI6ICJgYmFycGxvdGAgYWdyZWdhIChwcm9tZWRpYSkgbGEgcHJvcGluYSBkZSBjYWRhIGTDrWEgeSBkaWJ1amEgdW5hIGJhcnJhIHBvciBkw61hOyBgZXJyb3JiYXI9Tm9uZWAgcXVpdGEgZWwgaW50ZXJ2YWxvLiIsICJwc2V1ZG8iOiAiYmFycGxvdChkYXRhPXByb3BpbmFzLCB4PSdkaWEnLCB5PSdwcm9waW5hJywgb3JkZW4gSnVlIOKGkiBEb20sIHNpbiBiYXJyYXMgZGUgZXJyb3IpIn0sICJzbnMwOSI6IHsic29sdXRpb24iOiAic25zLmNvdW50cGxvdChkYXRhPXByb3BpbmFzLCB4PSdtb21lbnRvJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidjb3VudHBsb3QnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkucGF0Y2hlcykgPT0gMiBhbmQgc3VtKF9oZWlnaHRzKCkpID09IDEyMCIsICJjdXN0b21fbXNnIjogIlVzYSBzbnMuY291bnRwbG90IGNvbiB4PSdtb21lbnRvJyAoMiBiYXJyYXMgcXVlIHN1bWVuIDEyMCkuIn0sICJoaW50cyI6IFsiTGEgcHJlZ3VudGEgZXMgJ2N1w6FudG9zJywgbm8gJ2N1w6FudG8nLiIsICJObyBoYXkgdmFyaWFibGUgbnVtw6lyaWNhOiBzb2xvIHNlIGN1ZW50YW4gZmlsYXMgcG9yIGNhdGVnb3LDrWEuIiwgInNucy5jb3VudHBsb3QoZGF0YT0uLi4sIHg9Li4uKS4iXSwgImV4cGxhaW4iOiAiYGNvdW50cGxvdGAgY3VlbnRhIGN1w6FudGFzIGZpbGFzIGhheSBkZSBjYWRhIGNhdGVnb3LDrWEgKGVxdWl2YWxlIGEgdmFsdWVfY291bnRzIGdyYWZpY2FkbykuIiwgInBzZXVkbyI6ICJjb3VudHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycpIn0sICJzbnMtZTAyIjogeyJzb2x1dGlvbiI6ICJzbnMuYmFycGxvdChkYXRhPXByb3BpbmFzLCB4PSdkaWEnLCB5PSdwcm9waW5hJywgb3JkZXI9WydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10sIGVycm9yYmFyPU5vbmUpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInYmFycGxvdCcgaW4gX3BxX2NvZGUgYW5kIF94dGlja3MoKSA9PSBbJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSBhbmQgbGVuKF9heCgpLnBhdGNoZXMpID09IDQiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSA0IGJhcnJhcyBjb24gbG9zIGTDrWFzIGVuIGVsIG9yZGVuIEp1ZSwgVmllLCBTw6FiLCBEb20uIn0sICJoaW50cyI6IFsiTWlyw6EgZWwgZWplIFg6IMK/ZW4gcXXDqSBvcmRlbiBzYWxlbiBsb3MgZMOtYXM/IiwgIsK/RGUgZMOzbmRlIHNhY2Egc2VhYm9ybiBlbCBvcmRlbj8iLCAiSGF5IHVuIHBhcsOhbWV0cm8gcGFyYSBmaWphciBlbCBvcmRlbiBkZSBsYXMgY2F0ZWdvcsOtYXMuIl0sICJleHBsYWluIjogIlNpIGxhIGNvbHVtbmEgZXMgdGV4dG8sIHNlYWJvcm4gb3JkZW5hIGxhcyBjYXRlZ29yw61hcyAqKnNlZ8O6biBhcGFyZWNlbioqIGVuIGxhIHRhYmxhLiBQYXJhIGTDrWFzLCBtZXNlcyBvIHRyYW1vcywgZWwgb3JkZW4gbmF0dXJhbCBoYXkgcXVlIGluZGljYXJsbyBjb24gYG9yZGVyPVsuLi5dYCAobyBjb24gdW4gdGlwbyBjYXRlZ8OzcmljbyBvcmRlbmFkbykuIiwgInBzZXVkbyI6ICJiYXJwbG90KC4uLiwgb3JkZXI9WydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10sIGVycm9yYmFyPU5vbmUpIn0sICJzbnMxMCI6IHsic29sdXRpb24iOiAic25zLmJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nZGlhJywgeT0ncHJvcGluYScsIGh1ZT0nZnVtYWRvcicsIG9yZGVyPVsnSnVlJywgJ1ZpZScsICdTw6FiJywgJ0RvbSddKVxucGx0LnRpdGxlKCdQcm9waW5hcyBwb3IgZMOtYSB5IGZ1bWFkb3InKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2JveHBsb3QnIGluIF9wcV9jb2RlIGFuZCBfeHRpY2tzKCkgPT0gWydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10gYW5kIHNvcnRlZChfbGVnZW5kKCkpID09IFsnTm8nLCAnU8OtJ10gYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdQcm9waW5hcyBwb3IgZMOtYSB5IGZ1bWFkb3InIiwgImN1c3RvbV9tc2ciOiAiQm94cGxvdCBwb3IgZMOtYSAob3JkZW4gcGVkaWRvKSwgaHVlPSdmdW1hZG9yJyB5IGVsIHTDrXR1bG8gZXhhY3RvLiJ9LCAiaGludHMiOiBbIlVuYSBudW3DqXJpY2EsIGRvcyBjYXRlZ8OzcmljYXM6IHVuYSBlbiBYIHkgb3RyYSBlbiBodWUuIiwgIkJveHBsb3QgY29uIHg9J2RpYScsIHk9J3Byb3BpbmEnLCBodWU9J2Z1bWFkb3InLiIsICJBZ3JlZ8OhIGVsIG9yZGVuIHkgZWwgdMOtdHVsbyBjb24gcGx0LnRpdGxlLiJdLCAiZXhwbGFpbiI6ICJDYWRhIGTDrWEgbXVlc3RyYSBkb3MgY2FqYXMgKGZ1bWFkb3Igc8OtL25vKSwgbG8gcXVlIHBlcm1pdGUgY29tcGFyYXIgc3ViZ3J1cG9zIGRlbnRybyBkZSBjYWRhIGTDrWEuIiwgInBzZXVkbyI6ICJib3hwbG90KGRhdGE9cHJvcGluYXMsIHg9J2RpYScsIHk9J3Byb3BpbmEnLCBodWU9J2Z1bWFkb3InLCBvcmRlbiBKdWUg4oaSIERvbSlcbnTDrXR1bG8gJ1Byb3BpbmFzIHBvciBkw61hIHkgZnVtYWRvcicifSwgInNucy1mMDEiOiB7InNvbHV0aW9uIjogIm9yZGVuID0gWydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ11cbnNucy5ib3hwbG90KGRhdGE9cHJvcGluYXMsIHg9J2RpYScsIHk9J3Byb3BpbmEnLCBvcmRlcj1vcmRlbiwgY29sb3I9J3doaXRlJylcbnNucy5zdHJpcHBsb3QoZGF0YT1wcm9waW5hcywgeD0nZGlhJywgeT0ncHJvcGluYScsIG9yZGVyPW9yZGVuLCBhbHBoYT0wLjUpXG5wbHQudGl0bGUoJ1Byb3BpbmFzIHBvciBkw61hJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidib3hwbG90JyBpbiBfcHFfY29kZSBhbmQgJ3N0cmlwcGxvdCcgaW4gX3BxX2NvZGUgYW5kIF94dGlja3MoKSA9PSBbJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgdW4gYm94cGxvdCB5IHVuIHN0cmlwcGxvdCBwb3IgZMOtYSwgZW4gZWwgb3JkZW4gSnVlIOKGkiBEb20uIn0sICJoaW50cyI6IFsiRWplY3V0YWxvIHkgY29tZW50w6EgbGEgbMOtbmVhIGRlbCBzdHJpcHBsb3QgcGFyYSBjb21wYXJhci4iLCAiQ2FtYmnDoSBhbHBoYSBhIDEgeSBvYnNlcnbDoS4iLCAiUGVuc8OhIHF1w6kgbm8gdGUgZGljZSB1bmEgY2FqYS4iXSwgImV4cGxhaW4iOiAiU3VwZXJwb25lciBjYXBhcyAocmVzdW1lbiArIGRhdG9zIGluZGl2aWR1YWxlcykgZXMgdW5hIHByw6FjdGljYSBjb23Dum4gZW4gRURBIHBhcmEgbm8gb2N1bHRhciBlbCB0YW1hw7FvIG5pIGxhIGZvcm1hIHJlYWwgZGUgY2FkYSBncnVwby4iLCAicHNldWRvIjogIiJ9LCAic25zMTEiOiB7InNvbHV0aW9uIjogImNvcnIgPSBwcm9waW5hc1tbJ2N1ZW50YScsICdwcm9waW5hJywgJ3BlcnNvbmFzJ11dLmNvcnIoKVxuc25zLmhlYXRtYXAoY29yciwgYW5ub3Q9VHJ1ZSwgY21hcD0nY29vbHdhcm0nKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2hlYXRtYXAnIGluIF9wcV9jb2RlIGFuZCAnY29vbHdhcm0nIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkudGV4dHMpID09IDkiLCAiY3VzdG9tX21zZyI6ICJFbCBoZWF0bWFwIGRlYmUgc2VyIGRlIDN4MywgY29uIGFubm90PVRydWUgeSBjbWFwPSdjb29sd2FybScuIn0sICJoaW50cyI6IFsiUHJpbWVybyBjYWxjdWzDoSBsYSBtYXRyaXogZGUgY29ycmVsYWNpw7NuIGRlIGVzYXMgdHJlcyBjb2x1bW5hcy4iLCAiZGZbY29sc10uY29ycigpIGRldnVlbHZlIHVuYSBtYXRyaXogM8OXMy4iLCAiaGVhdG1hcCBjb24gYW5ub3Q9VHJ1ZSB5IGNtYXA9J2Nvb2x3YXJtJy4iXSwgImV4cGxhaW4iOiAiU2UgY2FsY3VsYSBsYSBjb3JyZWxhY2nDs24gZW50cmUgY2FkYSBwYXIgZGUgY29sdW1uYXMgeSBlbCBoZWF0bWFwIGxhIGNvbG9yZWEgKHJvam8vYXp1bCA9IHBvc2l0aXZhL25lZ2F0aXZhKSBjb24gbG9zIHZhbG9yZXMgZXNjcml0b3MuIiwgInBzZXVkbyI6ICJjb3JyIOKGkCBjb3JyZWxhY2nDs24gZGUgW2N1ZW50YSwgcHJvcGluYSwgcGVyc29uYXNdXG5oZWF0bWFwKGNvcnIsIHZhbG9yZXMgZXNjcml0b3MsIGNvbG9yZXMgJ2Nvb2x3YXJtJykifSwgInNucy1wMiI6IHsic29sdXRpb24iOiAic25zLmJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycsIHk9J3Byb3BpbmEnLCBvcmRlcj1bJ0FsbXVlcnpvJywgJ0NlbmEnXSlcbnBsdC50aXRsZSgnUHJvcGluYSBwb3IgbW9tZW50bycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInYm94cGxvdCcgaW4gX3BxX2NvZGUgYW5kIF94dGlja3MoKSA9PSBbJ0FsbXVlcnpvJywgJ0NlbmEnXSBhbmQgX2F4KCkuZ2V0X3RpdGxlKCkgPT0gJ1Byb3BpbmEgcG9yIG1vbWVudG8nIiwgImN1c3RvbV9tc2ciOiAiQm94cGxvdCBkZSBwcm9waW5hIHBvciBtb21lbnRvIChvcmRlbiBBbG11ZXJ6bywgQ2VuYSkgY29uIGVsIHTDrXR1bG8gJ1Byb3BpbmEgcG9yIG1vbWVudG8nLiJ9LCAiaGludHMiOiBbIkNhdGVnw7NyaWNhICsgbnVtw6lyaWNhOiDCv3F1w6kgZ3LDoWZpY28/IiwgIng9J21vbWVudG8nLCB5PSdwcm9waW5hJyB5IGVsIG9yZGVuIHBlZGlkby4iLCAiRWwgdMOtdHVsbyBzZSBhZ3JlZ2EgY29uIHBsdC50aXRsZS4iXSwgImV4cGxhaW4iOiAiQm94cGxvdCBkZSB1bmEgbnVtw6lyaWNhIHBvciBncnVwbzogcGVybWl0ZSBjb21wYXJhciBtZWRpYW5hcyB5IGRpc3BlcnNpw7NuIGVudHJlIGFsbXVlcnpvIHkgY2VuYS4iLCAicHNldWRvIjogImJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycsIHk9J3Byb3BpbmEnLCBvcmRlbiBbQWxtdWVyem8sIENlbmFdKVxudMOtdHVsbyJ9LCAic25zLXAzIjogeyJzb2x1dGlvbiI6ICJwcm9waW5hc1sncGN0J10gPSBwcm9waW5hc1sncHJvcGluYSddIC8gcHJvcGluYXNbJ2N1ZW50YSddICogMTAwXG5zbnMuc2NhdHRlcnBsb3QoZGF0YT1wcm9waW5hcywgeD0nY3VlbnRhJywgeT0ncHJvcGluYScsIGh1ZT0nbW9tZW50bycpXG5wbHQudGl0bGUoJ0N1ZW50YSB2cyBwcm9waW5hJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidzY2F0dGVycGxvdCcgaW4gX3BxX2NvZGUgYW5kICdwY3QnIGluIHByb3BpbmFzLmNvbHVtbnMgYW5kIGFicyhwcm9waW5hc1sncGN0J10ubWVhbigpIC0gKHByb3BpbmFzWydwcm9waW5hJ10gLyBwcm9waW5hc1snY3VlbnRhJ10gKiAxMDApLm1lYW4oKSkgPCAxZS05IGFuZCBzdW0obGVuKGMuZ2V0X29mZnNldHMoKSkgZm9yIGMgaW4gX2F4KCkuY29sbGVjdGlvbnMpID09IDEyMCBhbmQgX2F4KCkuZ2V0X3RpdGxlKCkgPT0gJ0N1ZW50YSB2cyBwcm9waW5hJyIsICJjdXN0b21fbXNnIjogIlNlIGVzcGVyYSBsYSBjb2x1bW5hIHBjdCAocHJvcGluYSAvIGN1ZW50YSDDlyAxMDApLCB1biBzY2F0dGVyIGNvbiBsb3MgMTIwIHB1bnRvcyBjb2xvcmVhZG8gcG9yIG1vbWVudG8geSBlbCB0w610dWxvICdDdWVudGEgdnMgcHJvcGluYScuIn0sICJoaW50cyI6IFsiU29uIGRvcyB0YXJlYXM6IHVuYSBjb2x1bW5hIG51ZXZhIHkgdW4gZ3LDoWZpY28uIiwgInBjdCA9IHByb3BpbmEgLyBjdWVudGEgw5cgMTAwLiIsICJzY2F0dGVycGxvdCBjb24gaHVlPSdtb21lbnRvJyB5IHTDrXR1bG8uIl0sICJleHBsYWluIjogIkxhIGNvbHVtbmEgcGN0IHBlcm1pdGUgY29tcGFyYXIgcHJvcGluYXMgcmVsYXRpdmFzIChpbmRlcGVuZGllbnRlcyBkZWwgdGFtYcOxbyBkZSBsYSBjdWVudGEpOyBlbCBzY2F0dGVyIGNvbiBodWUgbXVlc3RyYSBzaSBsYSByZWxhY2nDs24gZGlmaWVyZSBlbnRyZSBtb21lbnRvcy4iLCAicHNldWRvIjogInByb3BpbmFzWydwY3QnXSDihpAgcHJvcGluYSAvIGN1ZW50YSDDlyAxMDBcbnNjYXR0ZXJwbG90KGN1ZW50YSB2cyBwcm9waW5hLCBjb2xvciBwb3IgbW9tZW50byk7IHTDrXR1bG8ifSwgInNucy1wNCI6IHsic29sdXRpb24iOiAiY29yciA9IHByb3BpbmFzW1snY3VlbnRhJywgJ3Byb3BpbmEnLCAncGVyc29uYXMnXV0uY29ycigpXG5zbnMuaGVhdG1hcChjb3JyLCBhbm5vdD1UcnVlKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2hlYXRtYXAnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkudGV4dHMpID09IDkiLCAiY3VzdG9tX21zZyI6ICJFbCBoZWF0bWFwIGRlYmUgc2VyIDPDlzMgKGN1ZW50YSwgcHJvcGluYSwgcGVyc29uYXMpIGNvbiBsb3MgdmFsb3JlcyBlc2NyaXRvcy4ifSwgImhpbnRzIjogWyJMZcOpIGVsIGVycm9yOiDCv3F1w6kgdmFsb3Igbm8gcHVlZGUgY29udmVydGlyPyIsICLCv1F1w6kgY29sdW1uYXMgZGUgcHJvcGluYXMgc29uIHRleHRvPyIsICJTZWxlY2Npb27DoSBzb2xvIGxhcyBudW3DqXJpY2FzIGFudGVzIGRlIGNvcnIoKS4iXSwgImV4cGxhaW4iOiAiYGNvcnIoKWAgY2FsY3VsYSBjb3JyZWxhY2lvbmVzIGVudHJlIGNvbHVtbmFzICoqbnVtw6lyaWNhcyoqLiBDb24gY29sdW1uYXMgZGUgdGV4dG8gY29tbyAnSnVlJyBvICdDZW5hJywgbGFzIHZlcnNpb25lcyByZWNpZW50ZXMgZGUgcGFuZGFzIGxhbnphbiBlcnJvciAoYGNvdWxkIG5vdCBjb252ZXJ0IHN0cmluZyB0byBmbG9hdGApLiBIYXkgcXVlIHNlbGVjY2lvbmFyIGxhcyBjb2x1bW5hcyBudW3DqXJpY2FzIChvIHVzYXIgYG51bWVyaWNfb25seT1UcnVlYCkuIiwgInBzZXVkbyI6ICJjb3JyIOKGkCBjb3JyZWxhY2nDs24gZGUgW2N1ZW50YSwgcHJvcGluYSwgcGVyc29uYXNdXG5oZWF0bWFwKGNvcnIsIGFubm90PVRydWUpIn19'
_TABLAS = ['propinas', 'ventas_diarias']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Distribuciones

## 1. Hola, seaborn
`Código` · `Muy fácil`

¿Cuáles son los montos de cuenta más habituales? Dibujá la distribución de `cuenta` de `propinas` con seaborn.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

# sns y plt ya están importados


comprobar("sns01")

## 2. Histograma con curva de densidad
`Código` · `Medio`

Dibujá la distribución de `propina` con **15 intervalos** y la curva de densidad superpuesta (`kde=True`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

sns.histplot(data=propinas, x='propina')

comprobar("sns02")

## 3. Comparar distribuciones con hue
`Código` · `Medio`

¿Las cenas son más caras que los almuerzos? Compará la distribución de `cuenta` entre **Almuerzo** y **Cena** con curvas de densidad rellenas, una por momento.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns03")

## 4. hue sin tabla
`Debugging` · `Fácil`

Querés comparar la distribución de `cuenta` por `momento` con un histograma, pero este código da error. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

sns.histplot(propinas['cuenta'], hue='momento')

comprobar("sns-e01")

---
# 📌 Relaciones entre variables

## 5. Dispersión con colores
`Código` · `Medio`

¿La propina crece con la cuenta? ¿Es igual en almuerzos y cenas? Dibujá cada cuenta como un punto (`cuenta` en X, `propina` en Y) coloreado según `momento`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns04")

## 6. Línea de tendencia
`Código` · `Difícil`

Dibujá la relación entre `cuenta` (X) y `propina` (Y) con su **recta de tendencia**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns05")

## 7. Transferencia: personas y cuenta
`Transferencia` · `Medio`

Dibujá un scatterplot con `personas` en X, `cuenta` en Y y color según `fumador`.

> 🔁 **Qué cambia:** Antes relacionaste cuenta con propina. Ahora la pregunta es si las mesas con **más personas** gastan más, separando por fumador.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns-g01")

## 8. Tendencias en el tiempo
`Código` · `Medio`

Aplicá el estilo **"whitegrid"**, dibujá con seaborn la evolución de las `ventas` por `fecha` de `ventas_diarias` y poné el título **"Tendencia de ventas"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns06")

---
# 📌 Comparar grupos

## 9. Boxplot
`Código` · `Medio`

¿Qué día se dejan propinas más altas y más variables? Dibujá un boxplot de `propina` por `dia`, con los días en el orden `['Jue', 'Vie', 'Sáb', 'Dom']`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns07")

## 10. Transferencia: cuentas por momento
`Transferencia` · `Fácil`

Dibujá un boxplot de `cuenta` por `momento`, con el orden `['Almuerzo', 'Cena']`.

> 🔁 **Qué cambia:** Misma idea (comparar una numérica entre grupos), otras variables.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns-g02")

## 11. Barras con promedios
`Código` · `Difícil`

Dibujá la **propina media** por `dia` (orden `['Jue', 'Vie', 'Sáb', 'Dom']`) con barras y **sin barras de error**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns08")

## 12. Contar categorías
`Código` · `Muy fácil`

¿Cuántas cuentas hay de almuerzo y cuántas de cena? Graficalo con seaborn.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns09")

## 13. Los días desordenados
`Debugging` · `Fácil`

Este barplot de propina media por día aparece con los días en un orden confuso. Corregilo para que vayan de jueves a domingo (y sin barras de error).

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

sns.barplot(data=propinas, x='dia', y='propina', errorbar=None)

comprobar("sns-e02")

## 14. Dos variables categóricas
`Código` · `Difícil`

Implementá el plan: boxplot de `propina` por `dia` (orden `['Jue', 'Vie', 'Sáb', 'Dom']`) separado por `fumador`, con el título **"Propinas por día y fumador"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns10")

## 15. La IA combina dos gráficos
`IA copiloto` · `Medio`

Pediste 'mostrar las propinas por día viendo también cada dato individual'. La IA respondió con este código:

> 🤖 Prompt usado:
> Quiero comparar propinas por día con un boxplot, pero también ver cada propina individual para no esconder cuántos datos hay. Explicá qué hace cada capa.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

orden = ['Jue', 'Vie', 'Sáb', 'Dom']
sns.boxplot(data=propinas, x='dia', y='propina', order=orden, color='white')
sns.stripplot(data=propinas, x='dia', y='propina', order=orden, alpha=0.5)
plt.title('Propinas por día')

comprobar("sns-f01")

---
# 📌 Correlación

## 16. Mapa de calor de correlaciones
`Código` · `Difícil`

¿Qué variables numéricas se mueven juntas? Calculá la correlación entre `cuenta`, `propina` y `personas` y visualizala como mapa de calor, con los valores escritos (`annot=True`) y la paleta `"coolwarm"`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

corr = 


comprobar("sns11")

---
# 📌 Proyecto: análisis exploratorio visual de propinas

## 17. Fase 2 · Propina por momento
`Código` · `Medio`

Respondé P2: **¿cambia la propina entre almuerzo y cena?** Dibujá un boxplot de `propina` por `momento` (orden `['Almuerzo', 'Cena']`) con el título **"Propina por momento"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns-p2")

## 18. Fase 3 · Propina vs cuenta por momento
`Código` · `Difícil`

Respondé P3 con un matiz: **¿la relación entre cuenta y propina es igual en almuerzo y cena?** Dibujá un scatterplot de `cuenta` vs `propina` coloreado por `momento` y agregá la **propina como porcentaje** de la cuenta como columna `pct` en `propinas` (para usarla después). Título: **"Cuenta vs propina"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("sns-p3")

## 19. Fase 4 · El heatmap que no se dibuja
`Debugging` · `Medio`

Para P4, un compañero intentó dibujar el heatmap de correlación de toda la tabla y le da error. Corregilo usando solo las columnas numéricas `cuenta`, `propina` y `personas` (con `annot=True`).

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

corr = propinas.corr()
sns.heatmap(corr, annot=True)

comprobar("sns-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/pandas-quest/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.